# 2_Plata



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = 'dod7dumu05u6cn'
os.environ['DataZoneDomainId'] = 'dzd-crnhdsp0s7155z'
os.environ['DataZoneEnvironmentId'] = 'deeuj7l8vp1nzb'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "dod7dumu05u6cn",
                "DataZoneDomainId": "dzd-crnhdsp0s7155z",
                "DataZoneEnvironmentId": "deeuj7l8vp1nzb",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
import pandas as pd
import numpy as np
import boto3
import json
import awswrangler as wr
from datetime import datetime
import logging

# Configurar logging detallado
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ Librerías importadas")

✓ Librerías importadas


In [0]:
bronze_path = "s3://sin-exam-bronze-877617909831/"
silver_path = "s3://sin-exam-silver-877617909831/"
database = "sin-exam_db"

logger.info(f"Configuración cargada - Bronze: {bronze_path}, Silver: {silver_path}")

# Leer datos desde Bronze
s3_path = "s3://sin-exam-bronze-877617909831/raw/20251216_011426_Sample-Superstore.csv"
logger.info(f"Leyendo datos desde: {s3_path}")

df_bronze = wr.s3.read_csv(path=s3_path, encoding='cp1252')
logger.info(f"✓ Datos cargados: {len(df_bronze):,} filas - SampleSuperstore")

# Convertir fechas
df_bronze['Order Date'] = pd.to_datetime(df_bronze['Order Date'], format='%m/%d/%Y', errors='coerce')
df_bronze['Ship Date'] = pd.to_datetime(df_bronze['Ship Date'], format='%m/%d/%Y', errors='coerce')
logger.info("✓ Fechas convertidas a datetime")

In [0]:
class TransformationLogger:
    """Registra todas las transformaciones aplicadas"""
    
    def __init__(self):
        self.logs = []
        self.initial_rows = len(df_bronze)
        
    def log_transformation(self, step, action, rows_before, rows_after, details=None):
        transformation = {
            'step': step,
            'action': action,
            'rows_before': rows_before,
            'rows_after': rows_after,
            'rows_removed': rows_before - rows_after,
            'percentage_removed': ((rows_before - rows_after) / rows_before * 100) if rows_before > 0 else 0,
            'details': details or {}
        }
        self.logs.append(transformation)
        
        logger.info(f"TRANSFORMACIÓN: {action}")
        logger.info(f"  • Filas antes: {rows_before:,}")
        logger.info(f"  • Filas después: {rows_after:,}")
        logger.info(f"  • Filas eliminadas: {rows_before - rows_after:,} ({transformation['percentage_removed']:.2f}%)")
        if details:
            logger.info(f"  • Detalles: {details}")
        print("-" * 80)
    
    def get_summary(self):
        return pd.DataFrame(self.logs)

# Inicializar logger de transformaciones
transform_log = TransformationLogger()



In [0]:
print("="*80)
print("FASE 1: LIMPIEZA DE DATOS")
print("="*80)

df_clean = df_bronze.copy()
initial_rows = len(df_clean)

# 1. Eliminar duplicados completos
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
transform_log.log_transformation(
    step=1,
    action="Eliminación de filas duplicadas completas",
    rows_before=rows_before,
    rows_after=len(df_clean),
    details={'duplicates_found': rows_before - len(df_clean)}
)

# 2. Eliminar filas con valores nulos en columnas críticas
critical_columns = ['Order ID', 'Order Date', 'Customer ID', 'Sales']
rows_before = len(df_clean)
missing_before = df_clean[critical_columns].isnull().sum()
df_clean = df_clean.dropna(subset=critical_columns)
transform_log.log_transformation(
    step=2,
    action=f"Eliminación de filas con valores nulos en columnas críticas",
    rows_before=rows_before,
    rows_after=len(df_clean),
    details={
        'critical_columns': critical_columns,
        'missing_by_column': missing_before.to_dict()
    }
)

# 3. Validar rangos de datos
rows_before = len(df_clean)

# Ventas, Cantidad y Ganancia no pueden ser todas cero
invalid_rows = df_clean[
    (df_clean['Sales'] == 0) & 
    (df_clean['Quantity'] == 0) & 
    (df_clean['Profit'] == 0)
]
df_clean = df_clean.drop(invalid_rows.index)
logger.info(f"  • Eliminadas {len(invalid_rows)} filas con valores cero en todas las métricas")

# Cantidad debe ser mayor a 0
invalid_qty = df_clean[df_clean['Quantity'] <= 0]
df_clean = df_clean[df_clean['Quantity'] > 0]
logger.info(f"  • Eliminadas {len(invalid_qty)} filas con cantidad <= 0")

# Descuento debe estar entre 0 y 1 (o 0 y 100 si es porcentaje)
if df_clean['Discount'].max() > 1:
    df_clean['Discount'] = df_clean['Discount'] / 100
    logger.info("  • Descuentos convertidos de porcentaje a decimal")

transform_log.log_transformation(
    step=3,
    action="Validación de rangos de datos y corrección de valores inválidos",
    rows_before=rows_before,
    rows_after=len(df_clean),
    details={
        'invalid_metrics': len(invalid_rows),
        'invalid_quantity': len(invalid_qty)
    }
)

# 4. Imputación de valores faltantes no críticos
rows_before = len(df_clean)

# Postal Code: Imputar con 0 si falta
postal_missing = df_clean['Postal Code'].isnull().sum()
if postal_missing > 0:
    df_clean['Postal Code'].fillna(0, inplace=True)
    logger.info(f"  • Postal Code: {postal_missing} valores imputados con 0")

# State, City: Imputar con 'DESCONOCIDO'
for col in ['State', 'City']:
    missing_count = df_clean[col].isnull().sum()
    if missing_count > 0:
        df_clean[col].fillna('DESCONOCIDO', inplace=True)
        logger.info(f"  • {col}: {missing_count} valores imputados con 'DESCONOCIDO'")

transform_log.log_transformation(
    step=4,
    action="Imputación de valores faltantes en columnas no críticas",
    rows_before=rows_before,
    rows_after=len(df_clean),
    details={
        'postal_code_imputed': postal_missing,
        'method': 'Postal Code=0, State/City=DESCONOCIDO'
    }
)

# 5. Eliminar outliers extremos en métricas financieras (opcional - conservador)
rows_before = len(df_clean)
outliers_removed = 0

# Solo eliminar outliers muy extremos en Sales y Profit (5 IQR)
for col in ['Sales', 'Profit']:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 5 * IQR  # 5 IQR es muy conservador
    upper_bound = Q3 + 5 * IQR
    
    before = len(df_clean)
    df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    removed = before - len(df_clean)
    outliers_removed += removed
    
    if removed > 0:
        logger.info(f"  • {col}: {removed} outliers extremos eliminados (fuera de rango [{lower_bound:.2f}, {upper_bound:.2f}])")

transform_log.log_transformation(
    step=5,
    action="Eliminación de outliers extremos (5 IQR) - Conservador",
    rows_before=rows_before,
    rows_after=len(df_clean),
    details={
        'total_outliers': outliers_removed,
        'method': '5 IQR (muy conservador)',
        'columns': ['Sales', 'Profit']
    }
)

print(f"\n✓ LIMPIEZA COMPLETADA:")
print(f"  • Filas iniciales: {initial_rows:,}")
print(f"  • Filas finales: {len(df_clean):,}")
print(f"  • Filas eliminadas: {initial_rows - len(df_clean):,} ({(initial_rows - len(df_clean))/initial_rows*100:.2f}%)")
print(f"  • Tasa de retención: {len(df_clean)/initial_rows*100:.2f}%")



FASE 1: LIMPIEZA DE DATOS
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------

✓ LIMPIEZA COMPLETADA:
  • Filas iniciales: 9,994
  • Filas finales: 9,085
  • Filas eliminadas: 909 (9.10%)
  • Tasa de retención: 90.90%


In [0]:

print("\n" + "="*80)
print("FASE 2: CREACIÓN DE MODELO DIMENSIONAL (STAR SCHEMA)")
print("="*80)

# Extraer componentes temporales
df_clean['Year'] = df_clean['Order Date'].dt.year
df_clean['Month'] = df_clean['Order Date'].dt.month
df_clean['Day'] = df_clean['Order Date'].dt.day
df_clean['Quarter'] = df_clean['Order Date'].dt.quarter
df_clean['DayOfWeek'] = df_clean['Order Date'].dt.dayofweek
df_clean['MonthName'] = df_clean['Order Date'].dt.month_name()
df_clean['DayName'] = df_clean['Order Date'].dt.day_name()

# ============= DIMENSIÓN TIEMPO =============
logger.info("Creando dimensión tiempo...")

unique_dates = df_clean[['Order Date']].drop_duplicates()
dim_tiempo = pd.DataFrame({
    'fecha': unique_dates['Order Date'].values
})

dim_tiempo['tiempo_id'] = range(1, len(dim_tiempo) + 1)
dim_tiempo['año'] = dim_tiempo['fecha'].dt.year
dim_tiempo['mes'] = dim_tiempo['fecha'].dt.month
dim_tiempo['dia'] = dim_tiempo['fecha'].dt.day
dim_tiempo['trimestre'] = dim_tiempo['fecha'].dt.quarter
dim_tiempo['dia_semana'] = dim_tiempo['fecha'].dt.dayofweek
dim_tiempo['nombre_dia'] = dim_tiempo['fecha'].dt.day_name()
dim_tiempo['nombre_mes'] = dim_tiempo['fecha'].dt.month_name()
dim_tiempo['semana_año'] = dim_tiempo['fecha'].dt.isocalendar().week

# Reordenar columnas
dim_tiempo = dim_tiempo[['tiempo_id', 'fecha', 'año', 'mes', 'dia', 'trimestre', 
                          'dia_semana', 'nombre_dia', 'nombre_mes', 'semana_año']]

logger.info(f"✓ Dimensión Tiempo creada: {len(dim_tiempo)} registros")

# ============= DIMENSIÓN CLIENTE =============
logger.info("Creando dimensión cliente...")

dim_cliente = df_clean[['Customer ID', 'Customer Name', 'Segment']].drop_duplicates()
dim_cliente['cliente_id'] = range(1, len(dim_cliente) + 1)
dim_cliente = dim_cliente[['cliente_id', 'Customer ID', 'Customer Name', 'Segment']]
dim_cliente.columns = ['cliente_id', 'customer_id_original', 'nombre_cliente', 'segmento']

logger.info(f"✓ Dimensión Cliente creada: {len(dim_cliente)} registros")

# ============= DIMENSIÓN PRODUCTO =============
logger.info("Creando dimensión producto...")

dim_producto = df_clean[['Product ID', 'Product Name', 'Category', 'Sub-Category']].drop_duplicates()
dim_producto['producto_id'] = range(1, len(dim_producto) + 1)
dim_producto = dim_producto[['producto_id', 'Product ID', 'Product Name', 'Category', 'Sub-Category']]
dim_producto.columns = ['producto_id', 'product_id_original', 'nombre_producto', 'categoria', 'subcategoria']

logger.info(f"✓ Dimensión Producto creada: {len(dim_producto)} registros")

# ============= DIMENSIÓN UBICACIÓN =============
logger.info("Creando dimensión ubicación...")

dim_ubicacion = df_clean[['Country', 'Region', 'State', 'City', 'Postal Code']].drop_duplicates()
dim_ubicacion['ubicacion_id'] = range(1, len(dim_ubicacion) + 1)
dim_ubicacion = dim_ubicacion[['ubicacion_id', 'Country', 'Region', 'State', 'City', 'Postal Code']]
dim_ubicacion.columns = ['ubicacion_id', 'pais', 'region', 'estado', 'ciudad', 'codigo_postal']

logger.info(f"✓ Dimensión Ubicación creada: {len(dim_ubicacion)} registros")

# ============= DIMENSIÓN ENVÍO =============
logger.info("Creando dimensión envío...")

dim_envio = df_clean[['Ship Mode']].drop_duplicates()
dim_envio['envio_id'] = range(1, len(dim_envio) + 1)
dim_envio = dim_envio[['envio_id', 'Ship Mode']]
dim_envio.columns = ['envio_id', 'modo_envio']

logger.info(f"✓ Dimensión Envío creada: {len(dim_envio)} registros")

# ============= TABLA DE HECHOS =============
logger.info("Creando tabla de hechos...")

# Crear fact table con todas las métricas
fact_table = df_clean.copy()

# Join con dim_tiempo
fact_table = fact_table.merge(
    dim_tiempo[['tiempo_id', 'fecha']],
    left_on='Order Date',
    right_on='fecha',
    how='left'
).drop('fecha', axis=1)

# Join con dim_cliente
fact_table = fact_table.merge(
    dim_cliente[['cliente_id', 'customer_id_original']],
    left_on='Customer ID',
    right_on='customer_id_original',
    how='left'
).drop('customer_id_original', axis=1)

# Join con dim_producto
fact_table = fact_table.merge(
    dim_producto[['producto_id', 'product_id_original']],
    left_on='Product ID',
    right_on='product_id_original',
    how='left'
).drop('product_id_original', axis=1)

# Join con dim_ubicacion
fact_table = fact_table.merge(
    dim_ubicacion[['ubicacion_id', 'pais', 'region', 'estado', 'ciudad', 'codigo_postal']],
    left_on=['Country', 'Region', 'State', 'City', 'Postal Code'],
    right_on=['pais', 'region', 'estado', 'ciudad', 'codigo_postal'],
    how='left'
).drop(['pais', 'region', 'estado', 'ciudad', 'codigo_postal'], axis=1)

# Join con dim_envio
fact_table = fact_table.merge(
    dim_envio[['envio_id', 'modo_envio']],
    left_on='Ship Mode',
    right_on='modo_envio',
    how='left'
).drop('modo_envio', axis=1)

# Seleccionar solo columnas relevantes para fact table
fact_columns = [
    'Row ID',
    'Order ID',
    'tiempo_id',
    'cliente_id',
    'producto_id',
    'ubicacion_id',
    'envio_id',
    'Sales',
    'Quantity',
    'Discount',
    'Profit'
]

fact_table = fact_table[fact_columns]
fact_table.columns = [
    'row_id',
    'order_id',
    'tiempo_id',
    'cliente_id',
    'producto_id',
    'ubicacion_id',
    'envio_id',
    'ventas',
    'cantidad',
    'descuento',
    'ganancia'
]

logger.info(f"✓ Tabla de hechos creada: {len(fact_table)} registros, {len(fact_table.columns)} columnas")

print("\n📊 RESUMEN DEL MODELO DIMENSIONAL:")
print(f"  • Dimensión Tiempo: {len(dim_tiempo):,} registros")
print(f"  • Dimensión Cliente: {len(dim_cliente):,} registros")
print(f"  • Dimensión Producto: {len(dim_producto):,} registros")
print(f"  • Dimensión Ubicación: {len(dim_ubicacion):,} registros")
print(f"  • Dimensión Envío: {len(dim_envio):,} registros")
print(f"  • Tabla de Hechos: {len(fact_table):,} registros")
print(f"\n  📐 STAR SCHEMA:")
print(f"  • Total dimensiones: 5")
print(f"  • Métricas en fact table: 4 (ventas, cantidad, descuento, ganancia)")
print(f"  • Granularidad: Transaccional (una fila = una orden)")




FASE 2: CREACIÓN DE MODELO DIMENSIONAL (STAR SCHEMA)

📊 RESUMEN DEL MODELO DIMENSIONAL:
  • Dimensión Tiempo: 1,222 registros
  • Dimensión Cliente: 791 registros
  • Dimensión Producto: 1,816 registros
  • Dimensión Ubicación: 628 registros
  • Dimensión Envío: 4 registros
  • Tabla de Hechos: 9,358 registros

  📐 STAR SCHEMA:
  • Total dimensiones: 5
  • Métricas en fact table: 4 (ventas, cantidad, descuento, ganancia)
  • Granularidad: Transaccional (una fila = una orden)


In [0]:
print("\n" + "="*80)
print("FASE 3: GUARDANDO TABLAS EN CAPA SILVER")
print("="*80)

# Guardar dimensión tiempo
path_tiempo = f"{silver_path}dim_tiempo/"
wr.s3.to_parquet(
    df=dim_tiempo,
    path=path_tiempo,
    dataset=True,
    mode='overwrite',
    compression='snappy'
)
logger.info(f"✓ Dimensión Tiempo guardada en: {path_tiempo}")

# Guardar dimensión cliente
path_cliente = f"{silver_path}dim_cliente/"
wr.s3.to_parquet(
    df=dim_cliente,
    path=path_cliente,
    dataset=True,
    mode='overwrite',
    compression='snappy'
)
logger.info(f"✓ Dimensión Cliente guardada en: {path_cliente}")

# Guardar dimensión producto
path_producto = f"{silver_path}dim_producto/"
wr.s3.to_parquet(
    df=dim_producto,
    path=path_producto,
    dataset=True,
    mode='overwrite',
    compression='snappy'
)
logger.info(f"✓ Dimensión Producto guardada en: {path_producto}")

# Guardar dimensión ubicación
path_ubicacion = f"{silver_path}dim_ubicacion/"
wr.s3.to_parquet(
    df=dim_ubicacion,
    path=path_ubicacion,
    dataset=True,
    mode='overwrite',
    compression='snappy'
)
logger.info(f"✓ Dimensión Ubicación guardada en: {path_ubicacion}")

# Guardar dimensión envío
path_envio = f"{silver_path}dim_envio/"
wr.s3.to_parquet(
    df=dim_envio,
    path=path_envio,
    dataset=True,
    mode='overwrite',
    compression='snappy'
)
logger.info(f"✓ Dimensión Envío guardada en: {path_envio}")

# Guardar tabla de hechos (particionada por año para mejor performance)
path_fact = f"{silver_path}fact_ventas/"

# Agregar columna año para partición
fact_table_with_year = fact_table.merge(
    dim_tiempo[['tiempo_id', 'año']],
    on='tiempo_id',
    how='left'
)

wr.s3.to_parquet(
    df=fact_table_with_year,
    path=path_fact,
    dataset=True,
    mode='overwrite',
    compression='snappy',
    partition_cols=['año']  # Particionado por año para optimizar queries
)
logger.info(f"✓ Tabla de hechos guardada en: {path_fact} (particionada por año)")

print("\n📦 ALMACENAMIENTO EN SILVER:")
print(f"  • dim_tiempo: {len(dim_tiempo):,} registros")
print(f"  • dim_cliente: {len(dim_cliente):,} registros")
print(f"  • dim_producto: {len(dim_producto):,} registros")
print(f"  • dim_ubicacion: {len(dim_ubicacion):,} registros")
print(f"  • dim_envio: {len(dim_envio):,} registros")
print(f"  • fact_ventas: {len(fact_table):,} registros (particionado por año)")




FASE 3: GUARDANDO TABLAS EN CAPA SILVER



📦 ALMACENAMIENTO EN SILVER:
  • dim_tiempo: 1,222 registros
  • dim_cliente: 791 registros
  • dim_producto: 1,816 registros
  • dim_ubicacion: 628 registros
  • dim_envio: 4 registros
  • fact_ventas: 9,358 registros (particionado por año)


In [0]:
print("\n" + "="*80)
print("FASE 4: CATALOGANDO EN GLUE")
print("="*80)

glue = boto3.client('glue', region_name="us-east-1")
crawler_name = "sin-exam-silver-crawler"

logger.info(f"Ejecutando crawler: {crawler_name}")
try:
    glue.start_crawler(Name=crawler_name)
    logger.info("✓ Crawler iniciado - tardará 1-2 minutos")
except Exception as e:
    if 'CrawlerRunningException' in str(e):
        logger.warning("⚠️  Crawler ya está corriendo")
    else:
        logger.error(f"Error al iniciar crawler: {e}")




FASE 4: CATALOGANDO EN GLUE


In [0]:
print("\n" + "="*80)
print("GUARDANDO LOGS DE TRANSFORMACIÓN")
print("="*80)

# Resumen de transformaciones
transform_summary = transform_log.get_summary()
display(transform_summary)

# Guardar en S3 (carpeta silver)
s3 = boto3.client('s3')
log_json = transform_summary.to_json(orient='records', indent=2)
s3.put_object(
    Bucket="sin-exam-silver-877617909831",
    Key='silver/transformation_logs.json',
    Body=log_json,
    ContentType='application/json'
)

logger.info("✓ Logs guardados en S3")

# Resumen final
final_summary = {
    'timestamp': datetime.now().isoformat(),
    'bronze_rows': initial_rows,
    'silver_rows': len(df_clean),
    'rows_removed': initial_rows - len(df_clean),
    'percentage_removed': ((initial_rows - len(df_clean)) / initial_rows * 100),
    'dimensions_created': 5,
    'fact_table_rows': len(fact_table)
}

print("\n📊 RESUMEN FINAL:")
print(json.dumps(final_summary, indent=2))

print("\n" + "="*80)
print("TRANSFORMACIÓN A SILVER COMPLETADA")
print("="*80)
print("\n🎯 PRÓXIMOS PASOS:")
print("1. Espera 2 minutos a que el crawler termine")
print("2. Verifica en Athena que puedes consultar las tablas:")
print(f"   SELECT * FROM {database}.silver_fact_ventas LIMIT 10;")
print("3. Ejecuta el notebook 3_Oro.ipynb para crear KPIs")


GUARDANDO LOGS DE TRANSFORMACIÓN


,step,action,rows_before,rows_after,rows_removed,percentage_removed,details
0,1,Eliminación de filas duplicadas completas,9994,9994,0,0.000000,{'duplicates_found': 0}
1,2,Eliminación de filas con valores nulos en colu...,9994,9994,0,0.000000,"{'critical_columns': ['Order ID', 'Order Date'..."
2,3,Validación de rangos de datos y corrección de ...,9994,9994,0,0.000000,"{'invalid_metrics': 0, 'invalid_quantity': 0}"
3,4,Imputación de valores faltantes en columnas no...,9994,9994,0,0.000000,"{'postal_code_imputed': 0, 'method': 'Postal C..."
4,5,Eliminación de outliers extremos (5 IQR) - Con...,9994,9085,909,9.095457,"{'total_outliers': 909, 'method': '5 IQR (muy ..."



📊 RESUMEN FINAL:
{
  "timestamp": "2025-12-16T06:20:44.773879",
  "bronze_rows": 9994,
  "silver_rows": 9085,
  "rows_removed": 909,
  "percentage_removed": 9.095457274364618,
  "dimensions_created": 5,
  "fact_table_rows": 9358
}

TRANSFORMACIÓN A SILVER COMPLETADA

🎯 PRÓXIMOS PASOS:
1. Espera 2 minutos a que el crawler termine
2. Verifica en Athena que puedes consultar las tablas:
   SELECT * FROM sin-exam_db.silver_fact_ventas LIMIT 10;
3. Ejecuta el notebook 3_Oro.ipynb para crear KPIs


## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()